# Task 3: Dual-Image Delta Move

Chessboard understanding with Qwen3.5-0.8B and token-level patch reordering (REOrder). Train on Colab; every training run and evaluation is cached on the Hugging Face Hub, so re-running a cell skips work already done.

Downloading Libraries and Imports

In [1]:
!pip install -q --upgrade "pillow<11.0.0" "transformers>=4.45.0" accelerate scikit-learn tqdm chess cairosvg python-Levenshtein datasets peft huggingface_hub python-dotenv
!pip uninstall -y torchao -q
!pip install -q flash-linear-attention

import gc
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import login, snapshot_download
from peft import LoraConfig, PeftModel, TaskType
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 59.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━

Setting up environment

In [2]:
# Project configuration dictionary
CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

# Clone the repository on Colab, or use the local checkout
repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    if repo_root.exists():
        # /content survives a session restart, so update the existing clone
        print(f"Repository directory already exists at: {repo_root}; pulling latest changes...")
        pull = subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only"], capture_output=True, text=True)
        print((pull.stdout or pull.stderr).strip())
    else:
        auth_url = "https://"
        repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve().parent


print(f"Setup Complete. REPO_ROOT: {repo_root}")

# Check GPU and device availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Make the project modules importable
sys.path.insert(0, str(repo_root / "src"))

from eval.results import (
    load_or_init_results_table, append_or_replace, load_results_table, push_results_table, evaluate_with_order,
)
from eval.diagnostics import (
    headline_table, analyze_san_predictions, san_error_breakdown, frame_ablation_losses,
    training_health_report, backfill_cached_metrics, save_diagnostics,
)
from training.finetune import finetune_and_push_chessboard_model, experiment_repo_id, experiment_model_name
from training.learned_reordering import train_with_learned_reordering, describe_order, greedy_order
from training.reordering import resolve_square_order
from training.resolution import configure_image_resolution
from training.token_reordering import verify_token_reordering

# Hub namespace of every model, dataset and results table of the project
hf_org_prefix = "bdatm-project"

print("All custom modules and eval utilities imported successfully!")

Cloning repository from https://github.com/Aivon99/BigDataAndTextMiningProject.git...
Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject
Using device: cuda
GPU: Tesla T4
All custom modules and eval utilities imported successfully!


Downloading dataset for task 3 from HuggingFace repo

In [3]:
# Authenticate with Hugging Face:
# - Colab: reads a token from a Colab secret named HF_TOKEN (key icon, left sidebar)
# - Local: reads HF_TOKEN from a repo-root .env file, or the environment
# - Falls back to an interactive login prompt if neither is found
hf_token = None
if CONFIG["colab"]:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
else:
    from dotenv import load_dotenv
    load_dotenv(repo_root / ".env")
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub using HF_TOKEN.")
else:
    print("No HF_TOKEN found in secrets/.env/environment, falling back to interactive login.")
    login()

dataset_name = "bdatm-project/dataset_task3"

# Download the dataset repository once
print("Downloading dataset snapshot locally...")
local_data_dir = snapshot_download(
    repo_id=dataset_name,
    repo_type="dataset",
    local_dir="./my_dataset_files"
)

# Build the splits from the local copy: load_dataset(dataset_name) would download
# every image again and hit the Hugging Face rate limit.
print(f"Loading dataset '{dataset_name}' from the local snapshot...")
dataset_task3 = load_dataset("imagefolder", data_dir=local_data_dir, num_proc=16)

# Add the second frame (image_t1) to every split
print("Mapping the dataset across splits to include image_t1...")
for split_name in dataset_task3.keys():

    def add_second_frame(example):
        # Build the path incorporating the split folder (train/val/test)
        file_path = os.path.join(local_data_dir, split_name, example["file_name_t1"])
        example["image_t1"] = Image.open(file_path).convert("RGB")
        return example

    dataset_task3[split_name] = dataset_task3[split_name].map(add_second_frame, num_proc=4)

print("\nDone! Every split has been successfully mapped with image_t1.")
print(dataset_task3)

No HF_TOKEN found in secrets/.env/environment, falling back to interactive login.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8004 files:   0%|          | 0/8004 [00:00<?, ?it/s]

Loading dataset 'bdatm-project/dataset_task3' from the local snapshot...


Resolving data files:   0%|          | 0/6401 [00:00<?, ?it/s]

...

Mapping the dataset across splits to include image_t1...


Map (num_proc=4):   0%|          | 0/3200 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/400 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/400 [00:00<?, ? examples/s]


Done! Every split has been successfully mapped with image_t1.
DatasetDict({
    train: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1', 'image_t1'],
        num_rows: 3200
    })
    validation: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1', 'image_t1'],
        num_rows: 400
    })
    test: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1', 'image_t1'],
        num_rows: 400
    })
})


In [4]:
# Check one sample of the Task 3 dataset

print("Verifying dataset_task3 structure...")

print(f"Dataset splits available: {list(dataset_task3.keys())}")

sample_split = dataset_task3["train"]
print(f"Number of samples in train split: {len(sample_split)}")

sample = sample_split[0]

print("\n--- Sample Metadata ---")
print(f"Sample ID: {sample.get('sample_id', 'N/A')}")
print(f"Task type: {sample.get('task', 'N/A')}")
print(f"Ground Truth Move (Target): {sample.get('target', 'N/A')}")
print(f"Prompt preview:\n{sample.get('prompt', 'N/A')[:150]}...")

print("\n--- Image Verification ---")
img_t = sample.get("image")
img_t1 = sample.get("image_t1")

print(f"Frame 1 ('image'): {type(img_t)}")
if img_t is not None:
    print(f"  -> Size: {img_t.size}, Mode: {img_t.mode}")

print(f"Frame 2 ('image_t1'): {type(img_t1)}")
if img_t1 is not None:
    print(f"  -> Size: {img_t1.size}, Mode: {img_t1.mode}")

assert img_t is not None, "Error: Frame 1 ('image') is missing or None!"
assert img_t1 is not None, "Error: Frame 2 ('image_t1') is missing or None!"
assert "target" in sample, "Error: Target field is missing!"

print("\nVerification Successful! dataset_task3 is correctly formatted and ready for training/evaluation.")

Verifying dataset_task3 structure...
Dataset splits available: ['train', 'validation', 'test']
Number of samples in train split: 3200

--- Sample Metadata ---
Sample ID: sample_000000
Task type: task3
Ground Truth Move (Target): O-O
Prompt preview:
You are a specialized model for chessboard temporal reasoning.
Your goal is to identify the move that transitioned the chessboard from State t to Stat...

--- Image Verification ---
Frame 1 ('image'): <class 'PIL.PngImagePlugin.PngImageFile'>
  -> Size: (512, 512), Mode: RGB
Frame 2 ('image_t1'): <class 'PIL.PngImagePlugin.PngImageFile'>
  -> Size: (512, 512), Mode: RGB

Verification Successful! dataset_task3 is correctly formatted and ready for training/evaluation.


## Experiment setup

**Token-level patch reordering (REOrder).** A scan order is applied to the visual *tokens* inside the model: each image's token embeddings and their M-RoPE positions are permuted together (`src/training/token_reordering.py`), so the board image is never altered and every token keeps its true 2D position. Only the order in which the LLM reads them changes, which matters here because 18 of Qwen3.5's 24 LLM layers are Gated DeltaNet (a recurrent scan with no positional encoding).

The first runs of this notebook rearranged *pixel tiles* instead, which changed the board itself (a "file_wise" image was a transposed board). Their results stay in the results table, labelled `legacy: pixel-tile reordering, 512px`.

**Resolution (patch alignment).** At `IMAGE_SIZE = 256` the LLM sees an 8x8 token grid: exactly one token per chess square. The legacy 512px raster run is the resolution ablation (512px = 2x2 tokens per square).

In [5]:
IMAGE_SIZE = 256   # 256 -> 1 LLM token per square; 512 -> 2x2 tokens per square
TAG = f"tok{IMAGE_SIZE}"   # suffix of every Hub repo / results row of this setup
# Fixed scan orders compared with the raster baseline ("rank_wise" is raster itself).
# "hilbert" is one of REOrder's own orders.
STRATEGIES = ["zigzag", "spiral_center_out", "file_wise", "hilbert"]
N_SHOT = 3   # few-shot examples for the vanilla model, drawn from train
TASK = "task3"
model_id = "Qwen/Qwen3.5-0.8B"
hf_org_prefix = "bdatm-project"

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)


def load_base_model():
    return AutoModelForImageTextToText.from_pretrained(
        model_id, dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, device_map="auto"
    )


def free_gpu():
    """Call after `del`-ing the model variables, so their GPU memory is released."""
    gc.collect()
    torch.cuda.empty_cache()


for s in ["raster", *STRATEGIES]:
    print(f"{s} (reading step of each square):\n{describe_order(resolve_square_order(s))}\n")

raster (reading step of each square):
8  0  1  2  3  4  5  6  7
7  8  9 10 11 12 13 14 15
6 16 17 18 19 20 21 22 23
5 24 25 26 27 28 29 30 31
4 32 33 34 35 36 37 38 39
3 40 41 42 43 44 45 46 47
2 48 49 50 51 52 53 54 55
1 56 57 58 59 60 61 62 63
   a  b  c  d  e  f  g  h

zigzag (reading step of each square):
8  0  1  2  3  4  5  6  7
7 15 14 13 12 11 10  9  8
6 16 17 18 19 20 21 22 23
5 31 30 29 28 27 26 25 24
4 32 33 34 35 36 37 38 39
3 47 46 45 44 43 42 41 40
2 48 49 50 51 52 53 54 55
1 63 62 61 60 59 58 57 56
   a  b  c  d  e  f  g  h

spiral_center_out (reading step of each square):
8 63 62 61 60 59 58 57 56
7 36 35 34 33 32 31 30 55
6 37 16 15 14 13 12 29 54
5 38 17  4  3  2 11 28 53
4 39 18  5  0  1 10 27 52
3 40 19  6  7  8  9 26 51
2 41 20 21 22 23 24 25 50
1 42 43 44 45 46 47 48 49
   a  b  c  d  e  f  g  h

file_wise (reading step of each square):
8  0  8 16 24 32 40 48 56
7  1  9 17 25 33 41 49 57
6  2 10 18 26 34 42 50 58
5  3 11 19 27 35 43 51 59
4  4 12 20 28 36 44 52 60

## Vanilla model

The processor is set to `IMAGE_SIZE` once here; training, evaluation and generation all go through it. `verify_token_reordering` checks on this environment's `transformers` version that no-order output equals the unpatched model, that embeddings and positions move together, and that the order actually changes the output.

In [6]:
model_vanilla = load_base_model()
processor = AutoProcessor.from_pretrained(model_id)
configure_image_resolution(processor, IMAGE_SIZE)
verify_token_reordering(model_vanilla, processor, dataset_task3["test"][0]["image"], resolve_square_order("zigzag"))

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

...

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

[resolution] processor set to 256px: {'image_size': 256, 'patch_grid': (16, 16), 'llm_token_grid': (8, 8), 'tokens_per_square': 1}


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[token reordering] OK: {'image_tokens': 64, 'token_grid': (8, 8), 'max_logit_change': 2.0}


{'image_tokens': 64, 'token_grid': (8, 8), 'max_logit_change': 2.0}

In [7]:
# Grab the first test sample from the Task 3 dataset
test_sample = dataset_task3["test"][0]

fen = test_sample["fen"]
task_prompt = test_sample["prompt"]
ground_truth_move = test_sample["target"]  # Task 3 target is the SAN move
sample_id = test_sample["sample_id"]

# Frame 1: State t
frame_t_image = test_sample["image"]

# Frame 2: State t+1

frame_t_plus_1_image = test_sample["image_t1"]

print(f"Sample ID: {sample_id}")
print(f"FEN State: {fen}")
print(f"\n Prompt provided to the model:\n{task_prompt}\n")
print(f"Real Move (Ground Truth): {ground_truth_move}\n")

# Prepare the multimodal input format for the model with two ordered images (State t and State t+1)
chat_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": frame_t_image},
            {"type": "image", "image": frame_t_plus_1_image},
            {"type": "text", "text": task_prompt},
        ]
    }
]

# Apply the processor's chat template
formatted_text = processor.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

# Tokenize inputs and pass both images as a list to the processor, then move to the GPU device
model_inputs = processor(
    text=[formatted_text],
    images=[frame_t_image, frame_t_plus_1_image],
    padding=True,
    return_tensors="pt"
).to(device)

# Generate the zero-shot prediction for the delta move
print("Waiting for the model to respond....")
with torch.no_grad():
    output_token_ids = model_vanilla.generate(**model_inputs, max_new_tokens=128)

# Trim prompt tokens from the generated output
trimmed_output_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output_token_ids)
]
predicted_move_string = processor.batch_decode(
    trimmed_output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"\n MODEL OUTPUT: \n{predicted_move_string.strip()}")

Sample ID: sample_000000
FEN State: 6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - - 0 33

 Prompt provided to the model:
You are a specialized model for chessboard temporal reasoning.
Your goal is to identify the move that transitioned the chessboard from State t to State t+1.
Input:
- Frame 1: Chessboard state at time t.
- Frame 2: Chessboard state at time t+1.
Output Format:
Return only the move in Standard Algebraic Notation (SAN).

Real Move (Ground Truth): Bxe4

Waiting for the model to respond....


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.



 MODEL OUTPUT: 
The move is c4.


Zero-shot and few-shot evaluation of the vanilla model. Few-shot prepends `N_SHOT` solved train examples as earlier chat turns.

In [8]:
all_models_results = load_or_init_results_table(task=TASK, hf_org_prefix=hf_org_prefix)

vanilla_results_df, vanilla_summary_df = evaluate_with_order(
    TASK, model_vanilla, processor, dataset_task3["test"], hf_org_prefix=hf_org_prefix,
    model_name=experiment_model_name("Vanilla Qwen3.5-0.8B (Zero-Shot)", TAG),
)
all_models_results = append_or_replace(all_models_results, vanilla_summary_df)

few_shot_examples = [dataset_task3["train"][i] for i in range(N_SHOT)]
fewshot_results_df, fewshot_summary_df = evaluate_with_order(
    TASK, model_vanilla, processor, dataset_task3["test"], hf_org_prefix=hf_org_prefix,
    model_name=experiment_model_name(f"Vanilla Qwen3.5-0.8B ({N_SHOT}-Shot)", TAG),
    few_shot_examples=few_shot_examples,
)
all_models_results = append_or_replace(all_models_results, fewshot_summary_df)
display(fewshot_results_df.head() if fewshot_results_df is not None else fewshot_summary_df)
display(headline_table(all_models_results, TASK))
del model_vanilla
free_gpu()

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.51kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Loaded existing comparison table for task3 from 'bdatm-project/evaluation-results-task3' (21 model(s) already evaluated).
[task3] Using cached evaluation for 'Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256]', skipping inference.


(…).5-0.8b_%28zero-shot%29_%5Btok256%5D.csv:   0%|          | 0.00/36.8k [00:00<?, ?B/s]

[task3] Using cached evaluation for 'Vanilla Qwen3.5-0.8B (3-Shot) [tok256]', skipping inference.


(…)en3.5-0.8b_%283-shot%29_%5Btok256%5D.csv:   0%|          | 0.00/19.5k [00:00<?, ?B/s]

,sample_id,ground_truth,predicted,raw_output,exact_match,pred_san,em_extracted,multiline,legal,ambiguous,same_move,from_sq_match,to_sq_match,piece_match
0,sample_000000,Bxe4,Kc4,Kc4,0,Kc4,0,0,0,0,0,0,0,0
1,sample_000001,Rxe6,Kc4,Kc4,0,Kc4,0,0,0,0,0,0,0,0
2,sample_000002,Rd1,Kc4,Kc4,0,Kc4,0,0,0,0,0,0,0,0
3,sample_000003,h6,Kc4,Kc4,0,Kc4,0,0,0,0,0,0,0,0
4,sample_000004,Re1,Kc4,Kc4,0,Kc4,0,0,0,0,0,0,0,0


,model_name,setup,Exact Match,Same move (any notation),Legal move,EM raw output
0,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.8925,0.9675,0.9675,0.8875
1,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.8750,0.9650,0.9650,0.8750
2,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.8675,0.9525,0.9525,0.8675
3,Qwen + LoRA (Zigzag - SFT),"legacy: pixel-tile reordering, 512px",0.8650,0.9475,0.9475,0.3100
4,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.8050,0.9325,0.9375,0.8050
5,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.7975,0.8850,0.9025,0.7975
6,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.7750,0.8600,0.8700,0.7750
7,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.7075,0.8225,0.8400,0.7075
8,Qwen + LoRA (Hilbert - SFT) [tok256],"token reordering, 256px",0.6775,0.8375,0.8700,0.6775
9,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.5725,0.8100,0.8375,0.5725


## LoRA baseline (raster order)

Supervised LoRA fine-tuning with the native raster order (= rank-wise). Up to 10 epochs with early stopping on validation loss; the best epoch is kept, also across a Colab disconnect.

In [9]:
raster_repo = experiment_repo_id(TASK, "raster", TAG, hf_org_prefix)
trained = finetune_and_push_chessboard_model(
    "raster", dataset_task3, processor, load_base_model(), peft_config, TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
)
del trained
free_gpu()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]


TASK: TASK3 | STRATEGY: RASTER -> bdatm-project/qwen-task3-raster-tok256-lora


training_recipe.json:   0%|          | 0.00/33.0 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Found a resumable checkpoint on the Hub for 'bdatm-project/qwen-task3-raster-tok256-lora': /root/.cache/huggingface/hub/models--bdatm-project--qwen-task3-raster-tok256-lora/snapshots/f0250b6fc7ffec066ce927997a2b15862c85d79e/last-checkpoint
Resuming from /root/.cache/huggingface/hub/models--bdatm-project--qwen-task3-raster-tok256-lora/snapshots/f0250b6fc7ffec066ce927997a2b15862c85d79e/last-checkpoint
Training for 'bdatm-project/qwen-task3-raster-tok256-lora' already finished on the Hub, skipping training and loading its final adapter.


adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Nothing new to push for bdatm-project/qwen-task3-raster-tok256-lora.


In [10]:
raster_model = PeftModel.from_pretrained(load_base_model(), raster_repo)
raster_results_df, raster_summary_df = evaluate_with_order(
    TASK, raster_model, processor, dataset_task3["test"], hf_org_prefix=hf_org_prefix, adapter_repo_id=raster_repo,
    model_name=experiment_model_name("Qwen + LoRA (Raster - SFT)", TAG),
)
all_models_results = append_or_replace(all_models_results, raster_summary_df)
display(headline_table(all_models_results, TASK))

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task3] Using cached evaluation for 'Qwen + LoRA (Raster - SFT) [tok256]', skipping inference.


(…)lora_%28raster_-_sft%29_%5Btok256%5D.csv:   0%|          | 0.00/19.8k [00:00<?, ?B/s]

,model_name,setup,Exact Match,Same move (any notation),Legal move,EM raw output
0,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.8925,0.9675,0.9675,0.8875
1,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.8750,0.9650,0.9650,0.8750
2,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.8675,0.9525,0.9525,0.8675
3,Qwen + LoRA (Zigzag - SFT),"legacy: pixel-tile reordering, 512px",0.8650,0.9475,0.9475,0.3100
4,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.8050,0.9325,0.9375,0.8050
5,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.7975,0.8850,0.9025,0.7975
6,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.7750,0.8600,0.8700,0.7750
7,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.7075,0.8225,0.8400,0.7075
8,Qwen + LoRA (Hilbert - SFT) [tok256],"token reordering, 256px",0.6775,0.8375,0.8700,0.6775
9,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.5725,0.8100,0.8375,0.5725


## Training-free reordering

The raster-trained LoRA model reads the *same* images with a different token order, with no retraining. Since the board is intact, this measures how much the model depends on the order it was trained with.

In [11]:
for strat in STRATEGIES:
    _, summary = evaluate_with_order(
        TASK, raster_model, processor, dataset_task3["test"], order=strat, hf_org_prefix=hf_org_prefix,
        adapter_repo_id=raster_repo, model_name=experiment_model_name(f"Qwen + LoRA Raster, read {strat} (TF)", TAG),
    )
    all_models_results = append_or_replace(all_models_results, summary)
display(headline_table(all_models_results, TASK))

[task3] Using cached evaluation for 'Qwen + LoRA Raster, read zigzag (TF) [tok256]', skipping inference.


(…)2C_read_zigzag_%28tf%29_%5Btok256%5D.csv:   0%|          | 0.00/19.8k [00:00<?, ?B/s]

[task3] Using cached evaluation for 'Qwen + LoRA Raster, read spiral_center_out (TF) [tok256]', skipping inference.


(…)ral_center_out_%28tf%29_%5Btok256%5D.csv:   0%|          | 0.00/19.7k [00:00<?, ?B/s]

[task3] Using cached evaluation for 'Qwen + LoRA Raster, read file_wise (TF) [tok256]', skipping inference.


(…)read_file_wise_%28tf%29_%5Btok256%5D.csv:   0%|          | 0.00/19.8k [00:00<?, ?B/s]

[task3] Using cached evaluation for 'Qwen + LoRA Raster, read hilbert (TF) [tok256]', skipping inference.


(…)C_read_hilbert_%28tf%29_%5Btok256%5D.csv:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

,model_name,setup,Exact Match,Same move (any notation),Legal move,EM raw output
0,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.8925,0.9675,0.9675,0.8875
1,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.8750,0.9650,0.9650,0.8750
2,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.8675,0.9525,0.9525,0.8675
3,Qwen + LoRA (Zigzag - SFT),"legacy: pixel-tile reordering, 512px",0.8650,0.9475,0.9475,0.3100
4,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.8050,0.9325,0.9375,0.8050
5,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.7975,0.8850,0.9025,0.7975
6,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.7750,0.8600,0.8700,0.7750
7,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.7075,0.8225,0.8400,0.7075
8,Qwen + LoRA (Hilbert - SFT) [tok256],"token reordering, 256px",0.6775,0.8375,0.8700,0.6775
9,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.5725,0.8100,0.8375,0.5725


## Fine-tuning with each scan order (SFT)

One LoRA run per strategy, trained and evaluated with that token order. Each run loads a fresh base model and frees it afterwards.

In [12]:
for strat in STRATEGIES:
    trained = finetune_and_push_chessboard_model(
        strat, dataset_task3, processor, load_base_model(), peft_config, TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
    )
    del trained
    free_gpu()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]


TASK: TASK3 | STRATEGY: ZIGZAG -> bdatm-project/qwen-task3-zigzag-tok256-lora


training_recipe.json:   0%|          | 0.00/33.0 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

...

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Found a resumable checkpoint on the Hub for 'bdatm-project/qwen-task3-hilbert-tok256-lora': /root/.cache/huggingface/hub/models--bdatm-project--qwen-task3-hilbert-tok256-lora/snapshots/5392d71a47f607ecd30d548996bdd3ce04a9bff2/last-checkpoint
Resuming from /root/.cache/huggingface/hub/models--bdatm-project--qwen-task3-hilbert-tok256-lora/snapshots/5392d71a47f607ecd30d548996bdd3ce04a9bff2/last-checkpoint
Training for 'bdatm-project/qwen-task3-hilbert-tok256-lora' already finished on the Hub, skipping training and loading its final adapter.


adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Nothing new to push for bdatm-project/qwen-task3-hilbert-tok256-lora.


In [13]:
for strat in STRATEGIES:
    repo = experiment_repo_id(TASK, strat, TAG, hf_org_prefix)
    strat_model = PeftModel.from_pretrained(load_base_model(), repo)
    _, summary = evaluate_with_order(
        TASK, strat_model, processor, dataset_task3["test"], order=strat, hf_org_prefix=hf_org_prefix, adapter_repo_id=repo,
        model_name=experiment_model_name(f"Qwen + LoRA ({strat.capitalize()} - SFT)", TAG),
    )
    all_models_results = append_or_replace(all_models_results, summary)
    del strat_model
    free_gpu()
display(headline_table(all_models_results, TASK))

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task3] Using cached evaluation for 'Qwen + LoRA (Zigzag - SFT) [tok256]', skipping inference.


(…)lora_%28zigzag_-_sft%29_%5Btok256%5D.csv:   0%|          | 0.00/19.8k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task3] Using cached evaluation for 'Qwen + LoRA (Spiral_center_out - SFT) [tok256]', skipping inference.


...

(…)a_%28file_wise_-_sft%29_%5Btok256%5D.csv:   0%|          | 0.00/19.8k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task3] Using cached evaluation for 'Qwen + LoRA (Hilbert - SFT) [tok256]', skipping inference.


(…)ora_%28hilbert_-_sft%29_%5Btok256%5D.csv:   0%|          | 0.00/19.7k [00:00<?, ?B/s]

,model_name,setup,Exact Match,Same move (any notation),Legal move,EM raw output
0,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.8925,0.9675,0.9675,0.8875
1,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.8750,0.9650,0.9650,0.8750
2,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.8675,0.9525,0.9525,0.8675
3,Qwen + LoRA (Zigzag - SFT),"legacy: pixel-tile reordering, 512px",0.8650,0.9475,0.9475,0.3100
4,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.8050,0.9325,0.9375,0.8050
5,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.7975,0.8850,0.9025,0.7975
6,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.7750,0.8600,0.8700,0.7750
7,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.7075,0.8225,0.8400,0.7075
8,Qwen + LoRA (Hilbert - SFT) [tok256],"token reordering, 256px",0.6775,0.8375,0.8700,0.6775
9,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.5725,0.8100,0.8375,0.5725


## Learned reordering (REOrder)

REOrder's Plackett-Luce policy (`external/REOrder`) over the 64 squares, trained jointly with LoRA by REINFORCE (`src/training/learned_reordering.py`): one warm-up epoch in raster order, then the policy samples orders with a Gumbel temperature annealed 0.2 -> 0. Reward = minus the answer's cross-entropy; two orders are scored on each sample and compared with each other, so puzzle difficulty cancels out. Up to 10 epochs with early stopping on validation loss (patience 2); the best epoch is kept, together with the order it was validated with (the policy's greedy order, or raster if the warm-up epoch was best). Resumable from Hub checkpoints.

In [14]:
learned_model, learned_policy, learned_repo, learned_order = train_with_learned_reordering(
    dataset_task3, processor, load_base_model(), peft_config, task=TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
    num_train_epochs=10, policy_start_after_epochs=1,
)
print("Order used for evaluation (reading step of each square):")
print(describe_order(learned_order))
print("\nPolicy's greedy order:")
print(describe_order(greedy_order(learned_policy)))

_, learned_summary = evaluate_with_order(
    TASK, learned_model, processor, dataset_task3["test"], order=learned_order, hf_org_prefix=hf_org_prefix,
    adapter_repo_id=learned_repo, model_name=experiment_model_name("Qwen + LoRA (Learned Reordering)", TAG),
)
all_models_results = append_or_replace(all_models_results, learned_summary)
display(headline_table(all_models_results, TASK))
del learned_model
free_gpu()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

[task3] Resuming learned reordering at epoch 4, row 0 (optimizer step 1600).
[task3] epoch 4 step 1650: lm_loss=0.2001 |adv|=0.0610 temp=0.131


KeyboardInterrupt: 

## Resolution ablation

Same raster LoRA recipe at 512px (legacy run: 2x2 tokens per square) vs 256px (1 token per square).

In [ ]:
LEGACY_RASTER_512 = "Qwen + LoRA Fine-Tuning (from HF)"   # raster LoRA of the first runs, 512px images
rows = all_models_results["model_name"].isin([LEGACY_RASTER_512, experiment_model_name("Qwen + LoRA (Raster - SFT)", TAG)])
display(headline_table(all_models_results[rows], TASK))

## Results

Diagnostic columns are recomputed from the cached per-sample predictions (no inference), then the table is pushed to the Hub and reloaded from there. `headline_table` shows the main metrics first.

In [ ]:
all_models_results = backfill_cached_metrics(
    all_models_results, TASK, fens=dataset_task3["test"]["fen"], hf_org_prefix=hf_org_prefix
)
results_repo = f"{hf_org_prefix}/evaluation-results-{TASK}"
push_results_table(results_repo, all_models_results, commit_message=f"{TASK}: comparison table")
display(headline_table(load_results_table(results_repo), TASK))

## Diagnostics

`training_health_report` per run: eval-loss history, discontinuities, recipe markers, best vs published epoch.

In [ ]:
if CONFIG["colab"]:
    from google.colab import drive
    drive.mount("/content/drive")   # save_diagnostics writes here, surviving a runtime reset

health = {r: training_health_report(r) for r in [experiment_repo_id(TASK, s, TAG, hf_org_prefix) for s in ["raster", *STRATEGIES, "learned"]]}
health_df = pd.DataFrame([
    {"repo_id": r, "n_problems": len(h["problems"]), "problems": " | ".join(h["problems"])} for r, h in health.items()
])
display(health_df)
save_diagnostics(health_df, f"training_health_{TAG}", TASK)

Qualitative error analysis of the raster LoRA model: one failure mode per prediction (`same move, notation differs` = right move, written differently).

In [ ]:
analyzed = (raster_results_df if "same_move" in raster_results_df.columns
            else analyze_san_predictions(raster_results_df, dataset_task3["test"]["fen"]))
display(san_error_breakdown(analyzed))
save_diagnostics(analyzed, f"raster_lora_per_sample_{TAG}", TASK)

`frame_ablation_losses`: does the model compare the two frames? Teacher-forced answer loss with the frames correct, swapped or duplicated: `correct` must be clearly the lowest.

In [ ]:
ablation = frame_ablation_losses(raster_model, processor, dataset_task3["test"], n_samples=100)
save_diagnostics(ablation, f"frame_ablation_raster_{TAG}", TASK)